<a href="https://colab.research.google.com/github/kasturikirankumar1101-lab/AI_TOOLS/blob/main/RAGwithLangchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
✅ ✅ STEP 1: Install REQUIRED packages

!pip install -q \
langchain \
langchain-community \
langchain-openai \
langchain-chroma \
langchain-text-splitters \
chromadb \
pypdf


🔁 STEP 2: Restart runtime (VERY IMPORTANT)

import os
os.kill(os.getpid(), 9)

👉 If you skip this → errors will continue ❌


✅ STEP 3: Try import again

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

print("✅ langchain_community working")



In [1]:
# ingest.py
import os
import time
import json
import logging
from google.colab import drive, userdata as colab_userdata

from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# ── Setup ──────────────────────────────────────────────────────────────────────
drive.mount("/content/drive")

DATA_DIR   = "/content/drive/MyDrive/data"
CHROMA_DIR = "/content/chroma_db"
COLLECTION = "rag_chatbot"
STATE_FILE = "/content/file_state.json"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# ── API Key ────────────────────────────────────────────────────────────────────
api_key = colab_userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("❌ OPENAI_API_KEY not found in Colab Secrets.")

# ── Load / Save File State ─────────────────────────────────────────────────────
def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, "r") as f:
            return json.load(f)
    return {}

def save_state(state):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

file_state = load_state()

# ── Duplicate Detection ────────────────────────────────────────────────────────
# Maps lowercase filename → first full path seen, across both persisted state
# and the current run. Two files are duplicates when they share a filename
# but live at different paths.

def build_filename_index(state: dict) -> dict:
    """Return {basename: full_path} from persisted state."""
    index = {}
    for path in state:
        name = os.path.basename(path).lower()
        index.setdefault(name, path)
    return index

filename_index: dict = build_filename_index(file_state)


def check_duplicate(file_path: str) -> bool:
    """Return True and log a warning when a same-named file was already seen."""
    name     = os.path.basename(file_path).lower()
    existing = filename_index.get(name)
    if existing and os.path.abspath(existing) != os.path.abspath(file_path):
        logger.warning(
            "⚠️  DUPLICATE FILE DETECTED — '%s' already processed as '%s'. "
            "This file will NOT be processed as it is a duplicate.",
            os.path.basename(file_path), existing,
        )
        return True
    return False

# ── Initialize DB ──────────────────────────────────────────────────────────────
def get_db():
    embeddings = OpenAIEmbeddings(
        api_key=api_key,
        model="text-embedding-3-small",
        dimensions=1536,
    )
    return Chroma(
        collection_name=COLLECTION,
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR,
    )

# ── Splitter ───────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

# ── State helpers ──────────────────────────────────────────────────────────────
def should_process(file_path: str) -> bool:
    last_modified = os.path.getmtime(file_path)
    if file_path not in file_state:
        return True
    return file_state[file_path] != last_modified

def update_state(file_path: str):
    file_state[file_path] = os.path.getmtime(file_path)
    filename_index.setdefault(os.path.basename(file_path).lower(), file_path)
    save_state(file_state)

# ── Remove stale embeddings ────────────────────────────────────────────────────
def delete_old_embeddings(db, file_path: str):
    try:
        db._collection.delete(where={"source": file_path})
        logger.info("           🧹 Removed stale embeddings for: %s", os.path.basename(file_path))
    except Exception:
        pass  # No prior embeddings for this file — that's fine

# ── ASCII progress bar ─────────────────────────────────────────────────────────
def _progress_bar(done: int, total: int, width: int = 30) -> str:
    filled = int(width * done / total) if total else width
    bar    = "█" * filled + "░" * (width - filled)
    pct    = int(100 * done / total) if total else 100
    return f"[{bar}] {pct:3d}%  ({done}/{total} chunks)"

# ── Core ingestion logic ───────────────────────────────────────────────────────
def ingest_file(file_path: str, db):
    """Load, split, embed, and store a single PDF file with live progress."""

    logger.info("━" * 60)
    logger.info("📂 FILE    : %s", os.path.basename(file_path))

    # ── Duplicate guard ────────────────────────────────────────────────────────
    if check_duplicate(file_path):
        logger.info("━" * 60)
        return

    # ── Skip unchanged files ───────────────────────────────────────────────────
    if not should_process(file_path):
        logger.info("⏭️  SKIP    : No changes detected — already up to date.")
        logger.info("━" * 60)
        return

    start = time.time()

    # Step 1 — Load ────────────────────────────────────────────────────────────
    logger.info("📄 STEP 1/4 — Loading PDF …")
    try:
        docs = PyPDFLoader(file_path).load()
    except Exception as e:
        logger.error("❌ Failed to load '%s': %s", file_path, e)
        return
    logger.info("           ✅ Loaded %d page(s)", len(docs))

    # Step 2 — Split ───────────────────────────────────────────────────────────
    logger.info("✂️  STEP 2/4 — Splitting into chunks …")
    chunks = splitter.split_documents(
        [d for d in docs if d.page_content.strip()]
    )
    if not chunks:
        logger.warning("⚠️  No valid text content found — skipping file.")
        return
    logger.info("           ✅ Created %d chunk(s)", len(chunks))

    # Step 3 — Remove stale embeddings ─────────────────────────────────────────
    logger.info("🧹 STEP 3/4 — Removing stale embeddings (if any) …")
    delete_old_embeddings(db, file_path)

    for chunk in chunks:
        chunk.metadata["source"] = file_path

    # Step 4 — Embed & Store with live progress bar ────────────────────────────
    logger.info("🧠 STEP 4/4 — Generating embeddings & storing in ChromaDB …")

    BATCH = 50
    total = len(chunks)
    for i in range(0, total, BATCH):
        db.add_documents(chunks[i : i + BATCH])
        logger.info("           %s", _progress_bar(min(i + BATCH, total), total))

    # ── Summary ───────────────────────────────────────────────────────────────
    update_state(file_path)
    elapsed = round(time.time() - start, 2)

    logger.info("━" * 60)
    logger.info("🎉 COMPLETE : %s", os.path.basename(file_path))
    logger.info("⏱️  TIME     : %ss", elapsed)
    logger.info("📊 DB TOTAL : %d vector(s) in collection", db._collection.count())
    logger.info("━" * 60)

# ── Watchdog Handler ───────────────────────────────────────────────────────────
class NewFileHandler(FileSystemEventHandler):

    def __init__(self, db):
        self.db = db

    def on_created(self, event):
        if not event.is_directory and event.src_path.lower().endswith(".pdf"):
            logger.info("🆕 NEW FILE detected: %s", event.src_path)
            print("🆕 NEW FILE detected: %s", event.src_path)
            ingest_file(event.src_path, self.db)




    def on_modified(self, event):
        if not event.is_directory and event.src_path.lower().endswith(".pdf"):
            logger.info("✏️  MODIFIED FILE detected: %s", event.src_path)
            ingest_file(event.src_path, self.db)

# ── Initial Ingestion ──────────────────────────────────────────────────────────
def initial_ingestion(db):
    pdf_files = [
        os.path.join(root, f)
        for root, _, files in os.walk(DATA_DIR)
        for f in files
        if f.lower().endswith(".pdf")
    ]

    if not pdf_files:
        logger.info("ℹ️  No PDF files found in '%s'.", DATA_DIR)
        return

    logger.info("=" * 60)
    logger.info("🔄 INITIAL INGESTION — %d PDF file(s) found", len(pdf_files))
    logger.info("=" * 60)

    for idx, file_path in enumerate(pdf_files, start=1):
        logger.info("📁 File %d of %d", idx, len(pdf_files))
        ingest_file(file_path, db)
    print("=" * 60)
    print("✅ INITIAL INGESTION COMPLETE — %d file(s) scanned", len(pdf_files))
    print("=" * 60)
    logger.info("=" * 60)
    logger.info("✅ INITIAL INGESTION COMPLETE — %d file(s) scanned", len(pdf_files))
    logger.info("=" * 60)

# ── Start Watcher ──────────────────────────────────────────────────────────────
def start_watcher():
    db = get_db()
    initial_ingestion(db)

    event_handler = NewFileHandler(db)
    observer = Observer()
    observer.schedule(event_handler, DATA_DIR, recursive=True)
    observer.start()

    logger.info("👀 Watching '%s' for new/modified PDF files …", DATA_DIR)
    logger.info("   Press Ctrl+C to stop.\n")

    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        logger.info("🛑 Stopping watcher …")
        observer.stop()

    observer.join()
    logger.info("👋 Watcher stopped.")

# ── RUN ────────────────────────────────────────────────────────────────────────
start_watcher()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ INITIAL INGESTION COMPLETE — %d file(s) scanned 1
🆕 NEW FILE detected: %s /content/drive/MyDrive/data/1706.03762v7.pdf
